# Stability Analysis
## Symbolic to Numeric
This script will use Sympy. Taking my system to make the Jacobian [3*3]. From the jacobian, using jac sys eigenvalues, will give the symbolic eigen values, stored in eigs. then using a for loop of parameter over the eigs.sub. Return numbers.

In [4]:
## Import packaged
import sympy as sy
import itertools
import pandas as pd

### Easy example
Using example fromk https://math.colgate.edu/~wweckesser/math312Spring05/handouts/Linear2x2.pdf,
make sure of the understanding behind the method.

In [2]:

# Define symbolic variables
x, y, a, b, c, d = sy.symbols('x y a b c d')

# Define symbolic ODEs
dxdt = a*x + b*y
dydt = c*x + d*y

# Define Symbolic Jacobian with respect to all the parameters
f =sy.Matrix([dxdt, dydt])
print(f)

# Jacobian with respect to STATE VARIABLES N, I, S
J = f.jacobian([x,y])
print(J)

# calculating eigen values of jacobian
eig = J.eigenvals()
eig


Matrix([[a*x + b*y], [c*x + d*y]])
Matrix([[a, b], [c, d]])


{a/2 + d/2 - sqrt(a**2 - 2*a*d + 4*b*c + d**2)/2: 1,
 a/2 + d/2 + sqrt(a**2 - 2*a*d + 4*b*c + d**2)/2: 1}

In [3]:
new_eig = {}
for k, v in eig.items():
    new_key = k.subs({a:4, b:2, c:1, d:3})
    new_eig[new_key] = v

print(new_eig)

{2: 1, 5: 1}


In [4]:
# solving numerically
# initialise storage
new_eig = {}
#loop through a dictionary to substituite. convert to decimal
for k, v in eig.items(): # unpacking the dictionary that J.eigenvals() to get eigenvalue expressions and their multiplicities 
    new_key = k.subs({a:2, b:1, c:1, d:3}).evalf()# substitute vaules intoeigenvalue expressions .evalf() makes into decimal
    new_eig[new_key] = v # save the numeric eigen value and multiplicities

print(new_eig)

a_vals = (4, 4, 2)
b_vals = (2, -2, 1)
c_vals = (1, 3, 1)
d_vals = (3, -3, 3)

results = {}
for a_v, b_v, c_v, d_v in zip(a_vals, b_vals, c_vals, d_vals):
    new_eig = {}
    for k, v in eig.items():
        new_key = k.subs({a:a_v, b:b_v, c:c_v, d:d_v}).evalf()
        new_eig[new_key] = v
    results[(a_v, b_v, c_v, d_v)] = new_eig

print(results)

{1.38196601125011: 1, 3.61803398874989: 1}
{(4, 2, 1, 3): {2.00000000000000: 1, 5.00000000000000: 1}, (4, -2, 3, -3): {-2.00000000000000: 1, 3.00000000000000: 1}, (2, 1, 1, 3): {1.38196601125011: 1, 3.61803398874989: 1}}


# Neuroblastoma ODE

In [26]:
## Define the system symbolically
# Define symbolic variables
t, N, I, S, K, K1N, K1I, K1S, K2N, K2I, K2S, K3NS, K3NI, K3IN, K3IS, K3SI, K3SN, C = sy.symbols(
    't N I S K K1N K1I K1S K2N K2I K2S K3NS K3NI K3IN K3IS K3SI K3SN C'
)
# Define symbolic ODEs
dNdt = K1N * N * (1 - (N + I + S) / K) - K2N * N * C + K3SN * S + K3IN * I - K3NS * N - K3NI * N
dIdt = K1I * I * (1 - (N + I + S) / K) - K2I * I * C + K3SI * S + K3NI * N - K3IS * I - K3IN * I
dSdt = K1S * S * (1 - (N + I + S) / K) - K2S * S * C + K3NS * N + K3IS * I - K3SN * S - K3SI * S

# Define Symbolic Jacobian with respect to all the parameters
f =sy.Matrix([dNdt, dIdt, dSdt])
print(f)
# Jacobian with respect to STATE VARIABLES N, I, S
J = f.jacobian([N, I, S])
print(J)

# calculating eigen values of jacobian
eig = J.eigenvals()
eig


Matrix([[-C*K2N*N + I*K3IN + K1N*N*(1 - (I + N + S)/K) - K3NI*N - K3NS*N + K3SN*S], [-C*I*K2I + I*K1I*(1 - (I + N + S)/K) - I*K3IN - I*K3IS + K3NI*N + K3SI*S], [-C*K2S*S + I*K3IS + K1S*S*(1 - (I + N + S)/K) + K3NS*N - K3SI*S - K3SN*S]])
Matrix([[-C*K2N + K1N*(1 - (I + N + S)/K) - K3NI - K3NS - K1N*N/K, K3IN - K1N*N/K, K3SN - K1N*N/K], [-I*K1I/K + K3NI, -C*K2I - I*K1I/K + K1I*(1 - (I + N + S)/K) - K3IN - K3IS, -I*K1I/K + K3SI], [K3NS - K1S*S/K, K3IS - K1S*S/K, -C*K2S + K1S*(1 - (I + N + S)/K) - K3SI - K3SN - K1S*S/K]])


{-((C*K*K2I + C*K*K2N + C*K*K2S + 2*I*K1I + I*K1N + I*K1S - K*K1I - K*K1N - K*K1S + K*K3IN + K*K3IS + K*K3NI + K*K3NS + K*K3SI + K*K3SN + K1I*N + K1I*S + 2*K1N*N + K1N*S + K1S*N + 2*K1S*S)**2/K**2 - 3*(C**2*K**2*K2I*K2N + C**2*K**2*K2I*K2S + C**2*K**2*K2N*K2S + 2*C*I*K*K1I*K2N + 2*C*I*K*K1I*K2S + C*I*K*K1N*K2I + C*I*K*K1N*K2S + C*I*K*K1S*K2I + C*I*K*K1S*K2N - C*K**2*K1I*K2N - C*K**2*K1I*K2S - C*K**2*K1N*K2I - C*K**2*K1N*K2S - C*K**2*K1S*K2I - C*K**2*K1S*K2N + C*K**2*K2I*K3NI + C*K**2*K2I*K3NS + C*K**2*K2I*K3SI + C*K**2*K2I*K3SN + C*K**2*K2N*K3IN + C*K**2*K2N*K3IS + C*K**2*K2N*K3SI + C*K**2*K2N*K3SN + C*K**2*K2S*K3IN + C*K**2*K2S*K3IS + C*K**2*K2S*K3NI + C*K**2*K2S*K3NS + C*K*K1I*K2N*N + C*K*K1I*K2N*S + C*K*K1I*K2S*N + C*K*K1I*K2S*S + 2*C*K*K1N*K2I*N + C*K*K1N*K2I*S + 2*C*K*K1N*K2S*N + C*K*K1N*K2S*S + C*K*K1S*K2I*N + 2*C*K*K1S*K2I*S + C*K*K1S*K2N*N + 2*C*K*K1S*K2N*S + 2*I**2*K1I*K1N + 2*I**2*K1I*K1S + I**2*K1N*K1S - 3*I*K*K1I*K1N - 3*I*K*K1I*K1S + I*K*K1I*K3IN + I*K*K1I*K3IS + 2*I*K*K1I

Decising parameter sets for stability on NB model.\
K = 1\
K1N = 0.75, K1I = 0.5, K1S = 0.25,\
TargetK2 = 1, NonTargetK2 = [0, 0.5]\
K3 = [0.1, 0.5, 0.9]\
C = [0.1, 0.5, 0.9] \
N opt Patterns :\<N>|I>|S|, \<N>|I><S|\
S opt Patterns :|N|\<I>|S>,|N|\<I>|S|, |N|\<I>\<S>

## Symbolic

In [7]:
# PSUDO
# Next make for loop of parametres for N optimal pattern.
#Jacobian of the origin with regards to Drug. Keep drug undefined
# constant values 
#K : 1, K1N : 0.75, K1I : 0.5, K1S : 0.25,K2S : 1
# varying values
#K2I : [0,0.5], K2N :[0, 0.5], 
#imput k3 values based on N opt patterns  <N>|I>|S| & <N>|I><S|, | = 0 <>= K3s
#K3s[ 0.1, 0.5,0.9]
# Initial values
#N : 0, I:0, S:0
#J_numeric = J.subs({Dict})
#J_numeric

### Stability for N optimal patterns with S inhibitor at the origin with regards to Drug

In [8]:
import itertools
#### Stability for N optimal patterns with S inhibitor at the origin with regards to Drug
# Drug left symbolic, test combination of non target K2 and varied K3rate
# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2S: 1, # Carrying capacity, growth, target cell death
    N: 0, I: 0, S: 0 # ORIGIN
}

# Varying parameters
K2I_vals = [0, 0.5] # nontarget death
K2N_vals = [0, 0.5]
K3_vals  = [0.1, 0.5, 0.9]# interconversion rates

# K3 patterns
## Making functoin of optimal patterns with k3 values. | set to 0, <> to be filled with K3_vals 
def pattern_1(k3):
    # <N>|I>|S|
    return {K3NS: k3, K3NI: k3, K3IN: 0, K3IS: k3, K3SI: 0, K3SN: 0}

def pattern_2(k3):
    # <N>|I><S|
    return {K3NS: k3, K3NI: k3, K3IN: 0, K3IS: k3, K3SI: k3, K3SN: 0}
# dictionary of patterns functions
patterns = {
    'pattern_1': pattern_1,
    'pattern_2': pattern_2
}

# Build dictionary of dictionaries containing parameter sets
param_sets = {}
for k2i, k2n, k3, (pattern_name, pattern_func) in itertools.product( # go over all values for non target K2, K3s and patterns 
        K2I_vals, K2N_vals, K3_vals, patterns.items()):

    key = f'K2I={k2i}_K2N={k2n}_K3={k3}_{pattern_name}'

    param_sets[key] = {
        **fixed_params,
        K2I: k2i,
        K2N: k2n,
        **pattern_func(k3)# calling current patterns functiosn and filling in K3
    }

print(f'Total parameter combinations: {len(param_sets)}')

param_sets

Total parameter combinations: 24


{'K2I=0_K2N=0_K3=0.1_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  K3NS: 0.1,
  K3NI: 0.1,
  K3IN: 0,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0},
 'K2I=0_K2N=0_K3=0.1_pattern_2': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  K3NS: 0.1,
  K3NI: 0.1,
  K3IN: 0,
  K3IS: 0.1,
  K3SI: 0.1,
  K3SN: 0},
 'K2I=0_K2N=0_K3=0.5_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  K3NS: 0.5,
  K3NI: 0.5,
  K3IN: 0,
  K3IS: 0.5,
  K3SI: 0,
  K3SN: 0},
 'K2I=0_K2N=0_K3=0.5_pattern_2': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  K3NS: 0.5,
  K3NI: 0.5,
  K3IN: 0,
  K3IS: 0.5,
  K3SI: 0.5,
  K3SN: 0},
 'K2I=0_K2N=0_K3=0.9_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  K3NS: 0.9,
  K3NI: 0.9,
  K3IN: 0,
  K3IS:

In [13]:
# Get Jacobian for each parameter combination Drug stay symbolic
jacobian_Drug = {}
for key, params in param_sets.items():
    jacobian_Drug[key] = J.subs(params)
    print(f'{key}:')
    print(jacobian_Drug[key])
    print()

K2I=0_K2N=0_K3=0.1_pattern_1:
Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0], [0.100000000000000, 0.100000000000000, 0.25 - C]])

K2I=0_K2N=0_K3=0.1_pattern_2:
Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0.100000000000000], [0.100000000000000, 0.100000000000000, 0.15 - C]])

K2I=0_K2N=0_K3=0.5_pattern_1:
Matrix([[-0.250000000000000, 0, 0], [0.500000000000000, 0, 0], [0.500000000000000, 0.500000000000000, 0.25 - C]])

K2I=0_K2N=0_K3=0.5_pattern_2:
Matrix([[-0.250000000000000, 0, 0], [0.500000000000000, 0, 0.500000000000000], [0.500000000000000, 0.500000000000000, -C - 0.25]])

K2I=0_K2N=0_K3=0.9_pattern_1:
Matrix([[-1.05000000000000, 0, 0], [0.900000000000000, -0.400000000000000, 0], [0.900000000000000, 0.900000000000000, 0.25 - C]])

K2I=0_K2N=0_K3=0.9_pattern_2:
Matrix([[-1.05000000000000, 0, 0], [0.900000000000000, -0.400000000000000, 0.900000000000000], [0.900000000000000, 0.900000000000000, -C - 0.65]])

K2I=0_K2N=0.5_K3=0

In [34]:
# Get Jacobian and eigen values for each parameter combination Drug stay symbolic
results = {}
for key, params in param_sets.items():
    J_numeric = J.subs(params) # sub in numerical values to jacobian
    eigs = J_numeric.eigenvals() # calculate eigen values
    # convert to decimals
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jacobian_Drug[key]}')
    print(f'Eigen Values {results[key]}')   
    print()

K2I=0_K2N=0_K3=0.1_pattern_1
Jacobian Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0], [0.100000000000000, 0.100000000000000, 0.25 - C]])
Eigen Values {0.550000000000000: 1, 0.400000000000000: 1, 0.25 - 1.0*C: 1}

K2I=0_K2N=0_K3=0.1_pattern_2
Jacobian Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0.100000000000000], [0.100000000000000, 0.100000000000000, 0.15 - C]])
Eigen Values {0.550000000000000: 1, -0.5*C - 0.5*(1.0*C**2 + 0.5*C + 0.1025)**0.5 + 0.275: 1, -0.5*C + 0.5*(1.0*C**2 + 0.5*C + 0.1025)**0.5 + 0.275: 1}

K2I=0_K2N=0_K3=0.5_pattern_1
Jacobian Matrix([[-0.250000000000000, 0, 0], [0.500000000000000, 0, 0], [0.500000000000000, 0.500000000000000, 0.25 - C]])
Eigen Values {-0.250000000000000: 1, 0: 1, 0.25 - 1.0*C: 1}

K2I=0_K2N=0_K3=0.5_pattern_2
Jacobian Matrix([[-0.250000000000000, 0, 0], [0.500000000000000, 0, 0.500000000000000], [0.500000000000000, 0.500000000000000, -C - 0.25]])
Eigen Values {-0.250000000000000: 1, -0

In [35]:
import pandas as pd

rows = []

for key, eigs in results.items():

    # split key into columns

    parts = key.split('_')

    eig_list = list(eigs.keys())

    rows.append({

        'key': key,

        'K2I': parts[0].split('=')[1],

        'K2N': parts[1].split('=')[1],

        'K3':  parts[2].split('=')[1],

        'pattern': parts[3],

        'jacobian': str(jacobian_Drug[key]),  # save jacobian as string

        'eig_1': eig_list[0],

        'eig_2': eig_list[1],

        'eig_3': eig_list[2],

    })

df = pd.DataFrame(rows)

df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_NpatSinhibByDrug.csv', index=False)

print(df)

                                 key  K2I  K2N   K3  pattern  \
0       K2I=0_K2N=0_K3=0.1_pattern_1    0    0  0.1  pattern   
1       K2I=0_K2N=0_K3=0.1_pattern_2    0    0  0.1  pattern   
2       K2I=0_K2N=0_K3=0.5_pattern_1    0    0  0.5  pattern   
3       K2I=0_K2N=0_K3=0.5_pattern_2    0    0  0.5  pattern   
4       K2I=0_K2N=0_K3=0.9_pattern_1    0    0  0.9  pattern   
5       K2I=0_K2N=0_K3=0.9_pattern_2    0    0  0.9  pattern   
6     K2I=0_K2N=0.5_K3=0.1_pattern_1    0  0.5  0.1  pattern   
7     K2I=0_K2N=0.5_K3=0.1_pattern_2    0  0.5  0.1  pattern   
8     K2I=0_K2N=0.5_K3=0.5_pattern_1    0  0.5  0.5  pattern   
9     K2I=0_K2N=0.5_K3=0.5_pattern_2    0  0.5  0.5  pattern   
10    K2I=0_K2N=0.5_K3=0.9_pattern_1    0  0.5  0.9  pattern   
11    K2I=0_K2N=0.5_K3=0.9_pattern_2    0  0.5  0.9  pattern   
12    K2I=0.5_K2N=0_K3=0.1_pattern_1  0.5    0  0.1  pattern   
13    K2I=0.5_K2N=0_K3=0.1_pattern_2  0.5    0  0.1  pattern   
14    K2I=0.5_K2N=0_K3=0.5_pattern_1  0.

### Stability for N optimal patterns with S inhibitor at the origin with regards to Interconversion Rate

In [19]:
#### Stability for N optimal patterns with S inhibitor at the origin with regards to Interconversion Rate
# K3 left symbolic, test combination of non target K2 and varied Drug
# | set to 0 <> remain symbolic
# Fixed parameters

# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2S: 1,
    N: 0, I: 0, S: 0
}

# Varying parameters
K2I_vals = [0, 0.5]
K2N_vals = [0, 0.5]
C_vals   = [0.1, 0.5, 0.9]

# K3 patterns - only substitute zero values, leave non-zero symbolic
K3_patterns_symbolic = {
    'pattern_1': {K3IN: 0, K3SI: 0, K3SN: 0},  # K3NS, K3NI, K3IS remain symbolic
    #<N>|I>|S|  
    'pattern_2': {K3IN: 0, K3SN: 0}              # K3NS, K3NI, K3IS, K3SI remain symbolic
    #<N>|I><S|
}

# Build dictionary of Jacobians
jacobian_k3 = {}
for k2i, k2n, c_val, (pattern_name, k3_zeros) in itertools.product( # go over all values for non target K2, drugs and patterns
        K2I_vals, K2N_vals, C_vals, K3_patterns_symbolic.items()):

    key = f'K2I={k2i}_K2N={k2n}_C={c_val}_{pattern_name}'

    params = {
        **fixed_params,
        K2I: k2i,
        K2N: k2n,
        C: c_val,
        **k3_zeros
    }

    jacobian_k3[key] = J.subs(params)
    print(f'{key}:')
    print(jacobian_k3[key])
    print()

print(f'Total combinations: {len(jacobian_k3)}')

K2I=0_K2N=0_C=0.1_pattern_1:
Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, 0], [K3NS, K3IS, 0.150000000000000]])

K2I=0_K2N=0_C=0.1_pattern_2:
Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, K3SI], [K3NS, K3IS, 0.15 - K3SI]])

K2I=0_K2N=0_C=0.5_pattern_1:
Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, 0], [K3NS, K3IS, -0.250000000000000]])

K2I=0_K2N=0_C=0.5_pattern_2:
Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, K3SI], [K3NS, K3IS, -K3SI - 0.25]])

K2I=0_K2N=0_C=0.9_pattern_1:
Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, 0], [K3NS, K3IS, -0.650000000000000]])

K2I=0_K2N=0_C=0.9_pattern_2:
Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, K3SI], [K3NS, K3IS, -K3SI - 0.65]])

K2I=0_K2N=0.5_C=0.1_pattern_1:
Matrix([[-K3NI - K3NS + 0.7, 0, 0], [K3NI, 0.5 - K3IS, 0], [K3NS, K3IS, 0.150000000000000]])

K2I=0_K2N=0.5_C=0.1_pattern_2:
Matrix([[-K3NI - K3NS + 0.7, 0, 0], [K3NI, 0.5 - K3IS, K3SI], [K3NS, K3IS, 0.15 - K3SI]])

K2I=0_K2N=0.5_

In [20]:
# Get Jacobian and eigen values for each parameter combination K3 stay symbolic
results = {}
for key, jac in jacobian_k3.items():  
    eigs = jac.eigenvals() # calculate eigen values
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jac}')           # jac comes directly from the loop
    print(f'Eigen Values {results[key]}')
    print()

K2I=0_K2N=0_C=0.1_pattern_1
Jacobian Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, 0], [K3NS, K3IS, 0.150000000000000]])
Eigen Values {-1.0*K3NI - 1.0*K3NS + 0.75: 1, 0.5 - 1.0*K3IS: 1, 0.150000000000000: 1}

K2I=0_K2N=0_C=0.1_pattern_2
Jacobian Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, K3SI], [K3NS, K3IS, 0.15 - K3SI]])
Eigen Values {-1.0*K3NI - 1.0*K3NS + 0.75: 1, -0.5*K3IS - 0.5*K3SI - 0.707106781186548*(0.5*K3IS**2 + 1.0*K3IS*K3SI - 0.35*K3IS + 0.5*K3SI**2 + 0.35*K3SI + 0.06125)**0.5 + 0.325: 1, -0.5*K3IS - 0.5*K3SI + 0.707106781186548*(0.5*K3IS**2 + 1.0*K3IS*K3SI - 0.35*K3IS + 0.5*K3SI**2 + 0.35*K3SI + 0.06125)**0.5 + 0.325: 1}

K2I=0_K2N=0_C=0.5_pattern_1
Jacobian Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, 0], [K3NS, K3IS, -0.250000000000000]])
Eigen Values {-1.0*K3NI - 1.0*K3NS + 0.75: 1, 0.5 - 1.0*K3IS: 1, -0.250000000000000: 1}

K2I=0_K2N=0_C=0.5_pattern_2
Jacobian Matrix([[-K3NI - K3NS + 0.75, 0, 0], [K3NI, 0.5 - K3IS, K3SI], [K3NS, K3IS, 

In [21]:
# Save out put
import pandas as pd

rows = []

for key, eigs in results.items():

    # split key into columns

    parts = key.split('_')

    eig_list = list(eigs.keys())

    rows.append({

        'key': key,

        'K2I': parts[0].split('=')[1],

        'K2N': parts[1].split('=')[1],

        'K3':  parts[2].split('=')[1],

        'pattern': parts[3],

        'jacobian': str(jacobian_k3[key]),  # save jacobian as string

        'eig_1': eig_list[0],

        'eig_2': eig_list[1],

        'eig_3': eig_list[2],

    })

df = pd.DataFrame(rows)

df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_NpatSinhibByK3.csv', index=False)

print(df)

                                key  K2I  K2N   K3  pattern  \
0       K2I=0_K2N=0_C=0.1_pattern_1    0    0  0.1  pattern   
1       K2I=0_K2N=0_C=0.1_pattern_2    0    0  0.1  pattern   
2       K2I=0_K2N=0_C=0.5_pattern_1    0    0  0.5  pattern   
3       K2I=0_K2N=0_C=0.5_pattern_2    0    0  0.5  pattern   
4       K2I=0_K2N=0_C=0.9_pattern_1    0    0  0.9  pattern   
5       K2I=0_K2N=0_C=0.9_pattern_2    0    0  0.9  pattern   
6     K2I=0_K2N=0.5_C=0.1_pattern_1    0  0.5  0.1  pattern   
7     K2I=0_K2N=0.5_C=0.1_pattern_2    0  0.5  0.1  pattern   
8     K2I=0_K2N=0.5_C=0.5_pattern_1    0  0.5  0.5  pattern   
9     K2I=0_K2N=0.5_C=0.5_pattern_2    0  0.5  0.5  pattern   
10    K2I=0_K2N=0.5_C=0.9_pattern_1    0  0.5  0.9  pattern   
11    K2I=0_K2N=0.5_C=0.9_pattern_2    0  0.5  0.9  pattern   
12    K2I=0.5_K2N=0_C=0.1_pattern_1  0.5    0  0.1  pattern   
13    K2I=0.5_K2N=0_C=0.1_pattern_2  0.5    0  0.1  pattern   
14    K2I=0.5_K2N=0_C=0.5_pattern_1  0.5    0  0.5  pat

### Stability for S optimal patterns with N inhibitor at the origin with regards to Drug

In [10]:
#### Stability for S optimal patterns with N inhibitor at the origin with regards to Drug
# Drug left symbolic, test combination of non target K2 and varied K3rate
# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2N: 1, # Carrying capacity, growth, target cell death
    N: 0, I: 0, S: 0 # ORIGIN
}

# Varying parameters
K2I_vals = [0, 0.5] # nontarget death
K2S_vals = [0, 0.5]
K3_vals  = [0.1, 0.5, 0.9]# interconversion rates

# K3 patterns
## Making functoin of optimal patterns with k3 values. | set to 0, <> to be filled with K3_vals 
# |N|<I>|S>, |N|<I>|S|, |N|<I><S>
def pattern_3(k3):
    # |N|<I>|S>
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: 0, K3SN: k3}

def pattern_4(k3):
    # |N|<I>|S|
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: 0, K3SN: 0}

def pattern_5(k3):
    # |N|<I><S>
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: k3, K3SN: k3}
# dictionary of patterns functions
patterns = {
    'pattern_3': pattern_3,
    'pattern_4': pattern_4,
    'pattern_5': pattern_5
}

# Build dictionary of dictionaries containing parameter sets
jacobian_Durg = {}
for k2i, k2s, k3, (pattern_name, pattern_func) in itertools.product( # go over all values for non target K2, K3s and patterns 
        K2I_vals, K2S_vals, K3_vals, patterns.items()):

    key = f'K2I={k2i}_K2S={k2s}_K3={k3}_{pattern_name}'

    params  = {
        **fixed_params,
        K2I: k2i,
        K2S: k2s,
        **pattern_func(k3)# calling current patterns functiosn and filling in K3
        
    }
    jacobian_Durg[key] = J.subs(params)

print(f'Total parameter combinations: {len(jacobian_Durg)}')




Total parameter combinations: 36


In [11]:
# Get Jacobian and eigen values for each parameter combination Drug stay symbolic
results = {}
for key, jac in jacobian_Durg.items():  
    eigs = jac.eigenvals() # calculate eigen values
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jac}')           # jac comes directly from the loop
    print(f'Eigen Values {results[key]}')
    print()

K2I=0_K2S=0_K3=0.1_pattern_3
Jacobian Matrix([[0.75 - C, 0.100000000000000, 0.100000000000000], [0, 0.300000000000000, 0], [0, 0.100000000000000, 0.150000000000000]])
Eigen Values {0.300000000000000: 1, 0.150000000000000: 1, 0.75 - 1.0*C: 1}

K2I=0_K2S=0_K3=0.1_pattern_4
Jacobian Matrix([[0.75 - C, 0.100000000000000, 0], [0, 0.300000000000000, 0], [0, 0.100000000000000, 0.250000000000000]])
Eigen Values {0.300000000000000: 1, 0.75 - 1.0*C: 1, 0.250000000000000: 1}

K2I=0_K2S=0_K3=0.1_pattern_5
Jacobian Matrix([[0.75 - C, 0.100000000000000, 0.100000000000000], [0, 0.300000000000000, 0.100000000000000], [0, 0.100000000000000, 0.0500000000000000]])
Eigen Values {0.0149218940641788: 1, 0.335078105935821: 1, 0.75 - 1.0*C: 1}

K2I=0_K2S=0_K3=0.5_pattern_3
Jacobian Matrix([[0.75 - C, 0.500000000000000, 0.500000000000000], [0, -0.500000000000000, 0], [0, 0.500000000000000, -0.250000000000000]])
Eigen Values {-0.500000000000000: 1, -0.250000000000000: 1, 0.75 - 1.0*C: 1}

K2I=0_K2S=0_K3=0.5_pat

In [13]:
# Save out put
import pandas as pd

rows = []

for key, eigs in results.items():

    # split key into columns

    parts = key.split('_')

    eig_list = list(eigs.keys())

    rows.append({

        'key': key,

        'K2I': parts[0].split('=')[1],

        'K2N': parts[1].split('=')[1],

        'K3':  parts[2].split('=')[1],

        'pattern': parts[3],

        'jacobian': str(jacobian_Durg[key]),  # save jacobian as string

        'eig_1': eig_list[0],

        'eig_2': eig_list[1],

        'eig_3': eig_list[2],

    })

df = pd.DataFrame(rows)

df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_SpatNinhibByDrug.csv', index=False)

print(df)

                                 key  K2I  K2N   K3  pattern  \
0       K2I=0_K2S=0_K3=0.1_pattern_3    0    0  0.1  pattern   
1       K2I=0_K2S=0_K3=0.1_pattern_4    0    0  0.1  pattern   
2       K2I=0_K2S=0_K3=0.1_pattern_5    0    0  0.1  pattern   
3       K2I=0_K2S=0_K3=0.5_pattern_3    0    0  0.5  pattern   
4       K2I=0_K2S=0_K3=0.5_pattern_4    0    0  0.5  pattern   
5       K2I=0_K2S=0_K3=0.5_pattern_5    0    0  0.5  pattern   
6       K2I=0_K2S=0_K3=0.9_pattern_3    0    0  0.9  pattern   
7       K2I=0_K2S=0_K3=0.9_pattern_4    0    0  0.9  pattern   
8       K2I=0_K2S=0_K3=0.9_pattern_5    0    0  0.9  pattern   
9     K2I=0_K2S=0.5_K3=0.1_pattern_3    0  0.5  0.1  pattern   
10    K2I=0_K2S=0.5_K3=0.1_pattern_4    0  0.5  0.1  pattern   
11    K2I=0_K2S=0.5_K3=0.1_pattern_5    0  0.5  0.1  pattern   
12    K2I=0_K2S=0.5_K3=0.5_pattern_3    0  0.5  0.5  pattern   
13    K2I=0_K2S=0.5_K3=0.5_pattern_4    0  0.5  0.5  pattern   
14    K2I=0_K2S=0.5_K3=0.5_pattern_5    

### Stability for S optimal patterns with N inhibitor at the origin with regards to Interconversion Rate

In [22]:
#### Stability for N optimal patterns with S inhibitor at the origin with regards to Interconversion Rate
# K3 left symbolic, test combination of non target K2 and varied Drug
# | set to 0 <> remain symbolic
# Fixed parameters

# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2N: 1,
    N: 0, I: 0, S: 0
}

# Varying parameters
K2I_vals = [0, 0.5]
K2S_vals = [0, 0.5]
C_vals   = [0.1, 0.5, 0.9]


## Making functoin of optimal patterns. only substitute zero values, leave non-zero symbolic 
K3_patterns_symbolic = {
    'pattern_3': {K3NS: 0, K3NI: 0, K3SI: 0,},  
    # |N|<I>|S>
    'pattern_4': {K3NS: 0, K3NI: 0, K3SI: 0, K3SN: 0},     
    # |N|<I>|S|
    'pattern_5': {K3NS: 0, K3NI: 0 }
    # |N|<I><S>
}


# Build dictionary of Jacobians
jacobian_k3 = {}
for k2i, k2s, c_val, (pattern_name, k3_zeros) in itertools.product( # go over all values for non target K2, drugs and patterns
        K2I_vals, K2S_vals, C_vals, K3_patterns_symbolic.items()):

    key = f'K2I={k2i}_K2S={k2s}_C={c_val}_{pattern_name}'

    params = {
        **fixed_params,
        K2I: k2i,
        K2S: k2s,
        C: c_val,
        **k3_zeros
    }

    jacobian_k3[key] = J.subs(params)
    print(f'{key}:')
    print(jacobian_k3[key])
    print()

print(f'Total combinations: {len(jacobian_k3)}')

K2I=0_K2S=0_C=0.1_pattern_3:
Matrix([[0.650000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.25 - K3SN]])

K2I=0_K2S=0_C=0.1_pattern_4:
Matrix([[0.650000000000000, K3IN, 0], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.250000000000000]])

K2I=0_K2S=0_C=0.1_pattern_5:
Matrix([[0.650000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, K3SI], [0, K3IS, -K3SI - K3SN + 0.25]])

K2I=0_K2S=0_C=0.5_pattern_3:
Matrix([[0.250000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.25 - K3SN]])

K2I=0_K2S=0_C=0.5_pattern_4:
Matrix([[0.250000000000000, K3IN, 0], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.250000000000000]])

K2I=0_K2S=0_C=0.5_pattern_5:
Matrix([[0.250000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, K3SI], [0, K3IS, -K3SI - K3SN + 0.25]])

K2I=0_K2S=0_C=0.9_pattern_3:
Matrix([[-0.150000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.25 - K3SN]])

K2I=0_K2S=0_C=0.9_pattern_4:
Matrix([[-0.150000000000000, K3IN, 0], [0, -K3IN - K3IS + 0.5, 0], [0, K3I

In [23]:
# Get Jacobian and eigen values for each parameter combination K3 stay symbolic
results = {}
for key, jac in jacobian_k3.items():  
    eigs = jac.eigenvals() # calculate eigen values
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jac}')           # jac comes directly from the loop
    print(f'Eigen Values {results[key]}')
    print()

K2I=0_K2S=0_C=0.1_pattern_3
Jacobian Matrix([[0.650000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.25 - K3SN]])
Eigen Values {-1.0*K3IN - 1.0*K3IS + 0.5: 1, 0.25 - 1.0*K3SN: 1, 0.650000000000000: 1}

K2I=0_K2S=0_C=0.1_pattern_4
Jacobian Matrix([[0.650000000000000, K3IN, 0], [0, -K3IN - K3IS + 0.5, 0], [0, K3IS, 0.250000000000000]])
Eigen Values {-1.0*K3IN - 1.0*K3IS + 0.5: 1, 0.650000000000000: 1, 0.250000000000000: 1}

K2I=0_K2S=0_C=0.1_pattern_5
Jacobian Matrix([[0.650000000000000, K3IN, K3SN], [0, -K3IN - K3IS + 0.5, K3SI], [0, K3IS, -K3SI - K3SN + 0.25]])
Eigen Values {-0.5*K3IN - 0.5*K3IS - 0.5*K3SI - 0.5*K3SN - 0.707106781186548*(0.5*K3IN**2 + 1.0*K3IN*K3IS - 1.0*K3IN*K3SI - 1.0*K3IN*K3SN - 0.25*K3IN + 0.5*K3IS**2 + 1.0*K3IS*K3SI - 1.0*K3IS*K3SN - 0.25*K3IS + 0.5*K3SI**2 + 1.0*K3SI*K3SN + 0.25*K3SI + 0.5*K3SN**2 + 0.25*K3SN + 0.03125)**0.5 + 0.375: 1, -0.5*K3IN - 0.5*K3IS - 0.5*K3SI - 0.5*K3SN + 0.707106781186548*(0.5*K3IN**2 + 1.0*K3IN*K3IS - 1.0*K3IN*K3SI - 1

In [25]:
# Save output
import pandas as pd

rows = []

for key, eigs in results.items():

    # split key into columns

    
    # expand eigenvalues by multiplicity so we always get 3
    # if eigs = {λ₁: 1, λ₂: 2}
    # the loop produces:
    # eig_list = [λ₁, λ₂, λ₂]  # λ₂ appears twice because multiplicity = 2
    eig_list = []
    for eig_val, multiplicity in eigs.items():
        for _ in range(multiplicity):
            eig_list.append(eig_val)
    
    # pad with None if still less than 3 (safety net)
    while len(eig_list) < 3:
        eig_list.append(None)
    
    rows.append({
        'key': key,
        'K2I': parts[0].split('=')[1],
        'K2S': parts[1].split('=')[1],  # note: K2S not K2N for this analysis
        'C':   parts[2].split('=')[1],  # C not K3
        'pattern': parts[3],
        'jacobian': str(jacobian_k3[key]),
        'eig_1': str(eig_list[0]),
        'eig_2': str(eig_list[1]),
        'eig_3': str(eig_list[2]),
        'multiplicities': str({str(k): v for k, v in eigs.items()})
    })

df = pd.DataFrame(rows)
df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_SpatNinhibByK3.csv', index=False)
print(df)


                                key K2I K2S    C  pattern  \
0       K2I=0_K2S=0_C=0.1_pattern_3   0   0  0.5  pattern   
1       K2I=0_K2S=0_C=0.1_pattern_4   0   0  0.5  pattern   
2       K2I=0_K2S=0_C=0.1_pattern_5   0   0  0.5  pattern   
3       K2I=0_K2S=0_C=0.5_pattern_3   0   0  0.5  pattern   
4       K2I=0_K2S=0_C=0.5_pattern_4   0   0  0.5  pattern   
5       K2I=0_K2S=0_C=0.5_pattern_5   0   0  0.5  pattern   
6       K2I=0_K2S=0_C=0.9_pattern_3   0   0  0.5  pattern   
7       K2I=0_K2S=0_C=0.9_pattern_4   0   0  0.5  pattern   
8       K2I=0_K2S=0_C=0.9_pattern_5   0   0  0.5  pattern   
9     K2I=0_K2S=0.5_C=0.1_pattern_3   0   0  0.5  pattern   
10    K2I=0_K2S=0.5_C=0.1_pattern_4   0   0  0.5  pattern   
11    K2I=0_K2S=0.5_C=0.1_pattern_5   0   0  0.5  pattern   
12    K2I=0_K2S=0.5_C=0.5_pattern_3   0   0  0.5  pattern   
13    K2I=0_K2S=0.5_C=0.5_pattern_4   0   0  0.5  pattern   
14    K2I=0_K2S=0.5_C=0.5_pattern_5   0   0  0.5  pattern   
15    K2I=0_K2S=0.5_C=0.

## Numeric

In [27]:
# PSUDO
# Next make for loop of parametres for N optimal pattern.
# Jacobian of the origin .
# constant values 
#K : 1, K1N : 0.75, K1I : 0.5, K1S : 0.25,K2S : 1
# varying values
#K2I : [0,0.5], K2N :[0, 0.5], 
#imput k3 values based on N opt patterns  <N>|I>|S| & <N>|I><S|, | = 0 <>= K3s
#K3s[ 0.1, 0.5,0.9]
#Drugs[0.1,0.5,0.9]
# Initial values
#N : 0, I:0, S:0
#J_numeric = J.subs({Dict})
#J_numeric
#eig J_numeric
#save

### Stability for NTG at the origin

In [55]:
#### Stability for N optimal patterns with S inhibitor at the origin 
# test combination of non target K2 and varied K3rate and Drug
# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2N: 0, K2I:0, K2S: 0, # Carrying capacity, growth, target cell death
    N: 0, I: 0, S: 0 # ORIGIN
}

# Varying parameters
K3_vals  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]# interconversion rates
C_vals   = [0] # drug concentration

# K3 patterns
## Making functoin of optimal patterns with k3 values. | set to 0, <> to be filled with K3_vals 
def pattern_1(k3):
    # <N>|I>|S|
    return {K3NS: k3, K3NI: k3, K3IN: 0, K3IS: k3, K3SI: 0, K3SN: 0}

def pattern_2(k3):
    # <N>|I><S|
    return {K3NS: k3, K3NI: k3, K3IN: 0, K3IS: k3, K3SI: k3, K3SN: 0}

def pattern_3(k3):
    # |N|<I>|S>
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: 0, K3SN: k3}

def pattern_4(k3):
    # |N|<I>|S|
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: 0, K3SN: 0}

def pattern_5(k3):
    # |N|<I><S>
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: k3, K3SN: k3}
# dictionary of patterns functions
patterns = {
    'pattern_1': pattern_1,
    'pattern_2': pattern_2,
    'pattern_3': pattern_3,
    'pattern_4': pattern_4,
    'pattern_5': pattern_5
}

# Build dictionary of dictionaries containing parameter sets
param_sets = {}
for c_val, k3, (pattern_name, pattern_func) in itertools.product( # go over all values for non target K2, K3s and patterns 
        C_vals, K3_vals, patterns.items()):

    key = f'C={c_val}_K3={k3}_{pattern_name}'

    param_sets[key] = {
        **fixed_params,
        C: c_val,
        **pattern_func(k3)# calling current patterns functiosn and filling in K3
    }

print(f'Total parameter combinations: {len(param_sets)}')

param_sets

Total parameter combinations: 45


{'C=0_K3=0.1_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 0,
  K2I: 0,
  K2S: 0,
  N: 0,
  I: 0,
  S: 0,
  C: 0,
  K3NS: 0.1,
  K3NI: 0.1,
  K3IN: 0,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0},
 'C=0_K3=0.1_pattern_2': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 0,
  K2I: 0,
  K2S: 0,
  N: 0,
  I: 0,
  S: 0,
  C: 0,
  K3NS: 0.1,
  K3NI: 0.1,
  K3IN: 0,
  K3IS: 0.1,
  K3SI: 0.1,
  K3SN: 0},
 'C=0_K3=0.1_pattern_3': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 0,
  K2I: 0,
  K2S: 0,
  N: 0,
  I: 0,
  S: 0,
  C: 0,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.1,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0.1},
 'C=0_K3=0.1_pattern_4': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 0,
  K2I: 0,
  K2S: 0,
  N: 0,
  I: 0,
  S: 0,
  C: 0,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.1,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0},
 'C=0_K3=0.1_pattern_5': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 0,
  K2I: 0,
  K2S: 0,
  N: 0,
  I: 0,
  S: 0,
  C: 0,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.1,
  K3IS: 0.1,


In [56]:
# Get Jacobian and eigen values for each parameter combination 
jacobian_NTG = {}
results = {}
for key, params in param_sets.items():
    jacobian_NTG[key] = J.subs(params) # subing in parameter to systemn jacobian 
 
    J_numeric = J.subs(params) # sub in numerical values to jacobian
    eigs = J_numeric.eigenvals() # calculate eigen values
    # convert to decimals
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jacobian_NTG[key]}')
    print(f'Eigen Values {results[key]}')   
    print()

C=0_K3=0.1_pattern_1
Jacobian Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0], [0.100000000000000, 0.100000000000000, 0.250000000000000]])
Eigen Values {0.550000000000000: 1, 0.400000000000000: 1, 0.250000000000000: 1}

C=0_K3=0.1_pattern_2
Jacobian Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0.100000000000000], [0.100000000000000, 0.100000000000000, 0.150000000000000]])
Eigen Values {0.550000000000000: 1, 0.435078105935821: 1, 0.114921894064179: 1}

C=0_K3=0.1_pattern_3
Jacobian Matrix([[0.750000000000000, 0.100000000000000, 0.100000000000000], [0, 0.300000000000000, 0], [0, 0.100000000000000, 0.150000000000000]])
Eigen Values {0.750000000000000: 1, 0.300000000000000: 1, 0.150000000000000: 1}

C=0_K3=0.1_pattern_4
Jacobian Matrix([[0.750000000000000, 0.100000000000000, 0], [0, 0.300000000000000, 0], [0, 0.100000000000000, 0.250000000000000]])
Eigen Values {0.750000000000000: 1, 0.300000000000000: 1, 0.250000000000000: 1}

C=0_

In [57]:
# Save out put

rows = []

for key, eigs in results.items():

    # split key into columns
        # split key into columns

    
    # expand eigenvalues by multiplicity so we always get 3
    # if eigs = {λ₁: 1, λ₂: 2}
    # the loop produces:
    # eig_list = [λ₁, λ₂, λ₂]  # λ₂ appears twice because multiplicity = 2
    eig_list = []
    for eig_val, multiplicity in eigs.items():
        for _ in range(multiplicity):
            eig_list.append(eig_val)
    
    # pad with None if still less than 3 (safety net)
    while len(eig_list) < 3:
        eig_list.append(None)
    

    parts = key.split('_')

    rows.append({

        'key': key,
      
        'C':   parts[0].split('=')[1],  
        
        'K3':  parts[1].split('=')[1],        

        'pattern': 'pattern_' + key.split('_pattern_')[1],  # extract pattern reliably

        'jacobian': str(jacobian_NTG[key]),  # save jacobian as string

        'eig_1': eig_list[0],

        'eig_2': eig_list[1],

        'eig_3': eig_list[2],
        
        'multiplicities': str({str(k): v for k, v in eigs.items()})

    })

df = pd.DataFrame(rows)

df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_NTGFULL.csv', index=False)

print(df)

                     key  C   K3    pattern  \
0   C=0_K3=0.1_pattern_1  0  0.1  pattern_1   
1   C=0_K3=0.1_pattern_2  0  0.1  pattern_2   
2   C=0_K3=0.1_pattern_3  0  0.1  pattern_3   
3   C=0_K3=0.1_pattern_4  0  0.1  pattern_4   
4   C=0_K3=0.1_pattern_5  0  0.1  pattern_5   
5   C=0_K3=0.2_pattern_1  0  0.2  pattern_1   
6   C=0_K3=0.2_pattern_2  0  0.2  pattern_2   
7   C=0_K3=0.2_pattern_3  0  0.2  pattern_3   
8   C=0_K3=0.2_pattern_4  0  0.2  pattern_4   
9   C=0_K3=0.2_pattern_5  0  0.2  pattern_5   
10  C=0_K3=0.3_pattern_1  0  0.3  pattern_1   
11  C=0_K3=0.3_pattern_2  0  0.3  pattern_2   
12  C=0_K3=0.3_pattern_3  0  0.3  pattern_3   
13  C=0_K3=0.3_pattern_4  0  0.3  pattern_4   
14  C=0_K3=0.3_pattern_5  0  0.3  pattern_5   
15  C=0_K3=0.4_pattern_1  0  0.4  pattern_1   
16  C=0_K3=0.4_pattern_2  0  0.4  pattern_2   
17  C=0_K3=0.4_pattern_3  0  0.4  pattern_3   
18  C=0_K3=0.4_pattern_4  0  0.4  pattern_4   
19  C=0_K3=0.4_pattern_5  0  0.4  pattern_5   
20  C=0_K3=0.

### Stability for N optimal patterns with S inhibitor at the origin 

In [75]:
#### Stability for N optimal patterns with S inhibitor at the origin 
# test combination of non target K2 and varied K3rate and Drug
# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2S: 1, # Carrying capacity, growth, target cell death
    N: 0, I: 0, S: 0 # ORIGIN
}

# Varying parameters
K2I_vals = [0, 0.5] # nontarget death
K2N_vals = [0, 0.5]
K3_vals  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]# interconversion rates
C_vals   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] # drug concentration

# K3 patterns
## Making functoin of optimal patterns with k3 values. | set to 0, <> to be filled with K3_vals 
def pattern_1(k3):
    # <N>|I>|S|
    return {K3NS: k3, K3NI: k3, K3IN: 0, K3IS: k3, K3SI: 0, K3SN: 0}

def pattern_2(k3):
    # <N>|I><S|
    return {K3NS: k3, K3NI: k3, K3IN: 0, K3IS: k3, K3SI: k3, K3SN: 0}
# dictionary of patterns functions
patterns = {
    'pattern_1': pattern_1,
    'pattern_2': pattern_2
}

# Build dictionary of dictionaries containing parameter sets
param_sets = {}
for k2i, k2n, c_val, k3, (pattern_name, pattern_func) in itertools.product( # go over all values for non target K2, K3s and patterns 
        K2I_vals, K2N_vals, C_vals, K3_vals, patterns.items()):

    key = f'K2I={k2i}_K2N={k2n}_C={c_val}_K3={k3}_{pattern_name}'

    param_sets[key] = {
        **fixed_params,
        K2I: k2i,
        K2N: k2n,
        C: c_val,
        **pattern_func(k3)# calling current patterns functiosn and filling in K3
    }

print(f'Total parameter combinations: {len(param_sets)}')

param_sets

Total parameter combinations: 648


{'K2I=0_K2N=0_C=0.1_K3=0.1_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  C: 0.1,
  K3NS: 0.1,
  K3NI: 0.1,
  K3IN: 0,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0},
 'K2I=0_K2N=0_C=0.1_K3=0.1_pattern_2': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  C: 0.1,
  K3NS: 0.1,
  K3NI: 0.1,
  K3IN: 0,
  K3IS: 0.1,
  K3SI: 0.1,
  K3SN: 0},
 'K2I=0_K2N=0_C=0.1_K3=0.2_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  C: 0.1,
  K3NS: 0.2,
  K3NI: 0.2,
  K3IN: 0,
  K3IS: 0.2,
  K3SI: 0,
  K3SN: 0},
 'K2I=0_K2N=0_C=0.1_K3=0.2_pattern_2': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2N: 0,
  C: 0.1,
  K3NS: 0.2,
  K3NI: 0.2,
  K3IN: 0,
  K3IS: 0.2,
  K3SI: 0.2,
  K3SN: 0},
 'K2I=0_K2N=0_C=0.1_K3=0.3_pattern_1': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2S: 1,
  N: 0,
  I: 0,
  

In [76]:
# Get Jacobian and eigen values for each parameter combination 
jacobian_NoptSInhib = {}
results = {}
for key, params in param_sets.items():
    jacobian_NoptSInhib[key] = J.subs(params) # subing in parameter to systemn jacobian 
 
    J_numeric = J.subs(params) # sub in numerical values to jacobian
    eigs = J_numeric.eigenvals() # calculate eigen values
    # convert to decimals
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jacobian_NoptSInhib[key]}')
    print(f'Eigen Values {results[key]}')   
    print()

K2I=0_K2N=0_C=0.1_K3=0.1_pattern_1
Jacobian Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0], [0.100000000000000, 0.100000000000000, 0.150000000000000]])
Eigen Values {0.550000000000000: 1, 0.400000000000000: 1, 0.150000000000000: 1}

K2I=0_K2N=0_C=0.1_K3=0.1_pattern_2
Jacobian Matrix([[0.550000000000000, 0, 0], [0.100000000000000, 0.400000000000000, 0.100000000000000], [0.100000000000000, 0.100000000000000, 0.0500000000000000]])
Eigen Values {0.550000000000000: 1, 0.426556443707464: 1, 0.0234435562925362: 1}

K2I=0_K2N=0_C=0.1_K3=0.2_pattern_1
Jacobian Matrix([[0.350000000000000, 0, 0], [0.200000000000000, 0.300000000000000, 0], [0.200000000000000, 0.200000000000000, 0.150000000000000]])
Eigen Values {0.350000000000000: 1, 0.300000000000000: 1, 0.150000000000000: 1}

K2I=0_K2N=0_C=0.1_K3=0.2_pattern_2
Jacobian Matrix([[0.350000000000000, 0, 0], [0.200000000000000, 0.300000000000000, 0.200000000000000], [0.200000000000000, 0.200000000000000, -0.0500000000000

In [77]:
# Save out put

rows = []

for key, eigs in results.items():

    # split key into columns
        # split key into columns

    
    # expand eigenvalues by multiplicity so we always get 3
    # if eigs = {λ₁: 1, λ₂: 2}
    # the loop produces:
    # eig_list = [λ₁, λ₂, λ₂]  # λ₂ appears twice because multiplicity = 2
    eig_list = []
    for eig_val, multiplicity in eigs.items():
        for _ in range(multiplicity):
            eig_list.append(eig_val)
    
    # pad with None if still less than 3 (safety net)
    while len(eig_list) < 3:
        eig_list.append(None)
    

    parts = key.split('_')

    rows.append({

        'key': key,

        'K2I': parts[0].split('=')[1],

        'K2N': parts[1].split('=')[1],
        
        'C':   parts[2].split('=')[1],  
        
        'K3':  parts[3].split('=')[1],        

        'pattern': 'pattern_' + key.split('_pattern_')[1],  # ← reliable pattern extraction

        'jacobian': str(jacobian_NoptSInhib[key]),  # save jacobian as string

        'eig_1': eig_list[0],

        'eig_2': eig_list[1],

        'eig_3': eig_list[2],
        
        'multiplicities': str({str(k): v for k, v in eigs.items()})

    })

df = pd.DataFrame(rows)

df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_NpatSinhibFULL.csv', index=False)

print(df)

                                        key  K2I  K2N    C   K3    pattern  \
0        K2I=0_K2N=0_C=0.1_K3=0.1_pattern_1    0    0  0.1  0.1  pattern_1   
1        K2I=0_K2N=0_C=0.1_K3=0.1_pattern_2    0    0  0.1  0.1  pattern_2   
2        K2I=0_K2N=0_C=0.1_K3=0.2_pattern_1    0    0  0.1  0.2  pattern_1   
3        K2I=0_K2N=0_C=0.1_K3=0.2_pattern_2    0    0  0.1  0.2  pattern_2   
4        K2I=0_K2N=0_C=0.1_K3=0.3_pattern_1    0    0  0.1  0.3  pattern_1   
..                                      ...  ...  ...  ...  ...        ...   
643  K2I=0.5_K2N=0.5_C=0.9_K3=0.7_pattern_2  0.5  0.5  0.9  0.7  pattern_2   
644  K2I=0.5_K2N=0.5_C=0.9_K3=0.8_pattern_1  0.5  0.5  0.9  0.8  pattern_1   
645  K2I=0.5_K2N=0.5_C=0.9_K3=0.8_pattern_2  0.5  0.5  0.9  0.8  pattern_2   
646  K2I=0.5_K2N=0.5_C=0.9_K3=0.9_pattern_1  0.5  0.5  0.9  0.9  pattern_1   
647  K2I=0.5_K2N=0.5_C=0.9_K3=0.9_pattern_2  0.5  0.5  0.9  0.9  pattern_2   

                                              jacobian         

### Stability for S optimal patterns with N inhibitor at the origin 

In [82]:
#### Stability for N optimal patterns with S inhibitor at the origin 
# test combination of non target K2 and varied K3rate and Drug
# Fixed parameters
fixed_params = {
    K: 1, K1N: 0.75, K1I: 0.5, K1S: 0.25, K2N: 1, # Carrying capacity, growth, target cell death
    N: 0, I: 0, S: 0 # ORIGIN
}

# Varying parameters
K2I_vals = [0, 0.5] # nontarget death
K2S_vals = [0, 0.5]
K3_vals  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]# interconversion rates
C_vals   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] # drug concentration

# K3 patterns
## Making functoin of optimal patterns with k3 values. | set to 0, <> to be filled with K3_vals 
# |N|<I>|S>, |N|<I>|S|, |N|<I><S>
def pattern_3(k3):
    # |N|<I>|S>
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: 0, K3SN: k3}

def pattern_4(k3):
    # |N|<I>|S|
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: 0, K3SN: 0}

def pattern_5(k3):
    # |N|<I><S>
    return {K3NS: 0, K3NI: 0, K3IN: k3, K3IS: k3, K3SI: k3, K3SN: k3}
# dictionary of patterns functions
patterns = {
    'pattern_3': pattern_3,
    'pattern_4': pattern_4,
    'pattern_5': pattern_5
}


# Build dictionary of dictionaries containing parameter sets
param_sets = {}
for k2i, k2s, c_val, k3, (pattern_name, pattern_func) in itertools.product( # go over all values for non target K2, K3s and patterns 
        K2I_vals, K2S_vals, C_vals, K3_vals, patterns.items()):

    key = f'K2I={k2i}_K2S={k2s}_C={c_val}_K3={k3}_{pattern_name}'

    param_sets[key] = {
        **fixed_params,
        K2I: k2i,
        K2S: k2s,
        C: c_val,
        **pattern_func(k3)# calling current patterns functiosn and filling in K3
    }

print(f'Total parameter combinations: {len(param_sets)}')

param_sets

Total parameter combinations: 972


{'K2I=0_K2S=0_C=0.1_K3=0.1_pattern_3': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2S: 0,
  C: 0.1,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.1,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0.1},
 'K2I=0_K2S=0_C=0.1_K3=0.1_pattern_4': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2S: 0,
  C: 0.1,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.1,
  K3IS: 0.1,
  K3SI: 0,
  K3SN: 0},
 'K2I=0_K2S=0_C=0.1_K3=0.1_pattern_5': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2S: 0,
  C: 0.1,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.1,
  K3IS: 0.1,
  K3SI: 0.1,
  K3SN: 0.1},
 'K2I=0_K2S=0_C=0.1_K3=0.2_pattern_3': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 1,
  N: 0,
  I: 0,
  S: 0,
  K2I: 0,
  K2S: 0,
  C: 0.1,
  K3NS: 0,
  K3NI: 0,
  K3IN: 0.2,
  K3IS: 0.2,
  K3SI: 0,
  K3SN: 0.2},
 'K2I=0_K2S=0_C=0.1_K3=0.2_pattern_4': {K: 1,
  K1N: 0.75,
  K1I: 0.5,
  K1S: 0.25,
  K2N: 1,
  N: 0,
  I: 0,
  S: 0

In [83]:
# Get Jacobian and eigen values for each parameter combination 
jacobian_SoptNInhib = {}
results = {}
for key, params in param_sets.items():
    jacobian_SoptNInhib[key] = J.subs(params) # subing in parameter to systemn jacobian 
 
    J_numeric = J.subs(params) # sub in numerical values to jacobian
    eigs = J_numeric.eigenvals() # calculate eigen values
    # convert to decimals
    results[key] = {k.evalf(): v for k, v in eigs.items()}
    print(f'{key}')
    print(f'Jacobian {jacobian_SoptNInhib[key]}')
    print(f'Eigen Values {results[key]}')   
    print()

K2I=0_K2S=0_C=0.1_K3=0.1_pattern_3
Jacobian Matrix([[0.650000000000000, 0.100000000000000, 0.100000000000000], [0, 0.300000000000000, 0], [0, 0.100000000000000, 0.150000000000000]])
Eigen Values {0.650000000000000: 1, 0.300000000000000: 1, 0.150000000000000: 1}

K2I=0_K2S=0_C=0.1_K3=0.1_pattern_4
Jacobian Matrix([[0.650000000000000, 0.100000000000000, 0], [0, 0.300000000000000, 0], [0, 0.100000000000000, 0.250000000000000]])
Eigen Values {0.650000000000000: 1, 0.300000000000000: 1, 0.250000000000000: 1}

K2I=0_K2S=0_C=0.1_K3=0.1_pattern_5
Jacobian Matrix([[0.650000000000000, 0.100000000000000, 0.100000000000000], [0, 0.300000000000000, 0.100000000000000], [0, 0.100000000000000, 0.0500000000000000]])
Eigen Values {0.650000000000000: 1, 0.335078105935821: 1, 0.0149218940641788: 1}

K2I=0_K2S=0_C=0.1_K3=0.2_pattern_3
Jacobian Matrix([[0.650000000000000, 0.200000000000000, 0.200000000000000], [0, 0.100000000000000, 0], [0, 0.200000000000000, 0.0500000000000000]])
Eigen Values {0.6500000000

In [84]:
# Save out put

rows = []

for key, eigs in results.items():

    # split key into columns
        # split key into columns

    
    # expand eigenvalues by multiplicity so we always get 3
    # if eigs = {λ₁: 1, λ₂: 2}
    # the loop produces:
    # eig_list = [λ₁, λ₂, λ₂]  # λ₂ appears twice because multiplicity = 2
    eig_list = []
    for eig_val, multiplicity in eigs.items():
        for _ in range(multiplicity):
            eig_list.append(eig_val)
    
    # pad with None if still less than 3 (safety net)
    while len(eig_list) < 3:
        eig_list.append(None)
    

    parts = key.split('_')

    rows.append({

        'key': key,

        'K2I': parts[0].split('=')[1],

        'K2S': parts[1].split('=')[1],
        
        'C':   parts[2].split('=')[1],  
        
        'K3':  parts[3].split('=')[1],        
        
        'pattern': 'pattern_' + key.split('_pattern_')[1],  # ← reliable pattern extraction
        
        'jacobian': str(jacobian_SoptNInhib[key]),  # save jacobian as string

        'eig_1': eig_list[0],

        'eig_2': eig_list[1],

        'eig_3': eig_list[2],
        
        'multiplicities': str({str(k): v for k, v in eigs.items()})

    })

df = pd.DataFrame(rows)

df.to_csv('C:/users/892215/OneDrive - hull.ac.uk/Desktop/Year3/Experiment_3/Stability/NoteBookOut/stability_SpatNinhibFULL.csv', index=False)

print(df)

                                        key  K2I  K2S    C   K3    pattern  \
0        K2I=0_K2S=0_C=0.1_K3=0.1_pattern_3    0    0  0.1  0.1  pattern_3   
1        K2I=0_K2S=0_C=0.1_K3=0.1_pattern_4    0    0  0.1  0.1  pattern_4   
2        K2I=0_K2S=0_C=0.1_K3=0.1_pattern_5    0    0  0.1  0.1  pattern_5   
3        K2I=0_K2S=0_C=0.1_K3=0.2_pattern_3    0    0  0.1  0.2  pattern_3   
4        K2I=0_K2S=0_C=0.1_K3=0.2_pattern_4    0    0  0.1  0.2  pattern_4   
..                                      ...  ...  ...  ...  ...        ...   
967  K2I=0.5_K2S=0.5_C=0.9_K3=0.8_pattern_4  0.5  0.5  0.9  0.8  pattern_4   
968  K2I=0.5_K2S=0.5_C=0.9_K3=0.8_pattern_5  0.5  0.5  0.9  0.8  pattern_5   
969  K2I=0.5_K2S=0.5_C=0.9_K3=0.9_pattern_3  0.5  0.5  0.9  0.9  pattern_3   
970  K2I=0.5_K2S=0.5_C=0.9_K3=0.9_pattern_4  0.5  0.5  0.9  0.9  pattern_4   
971  K2I=0.5_K2S=0.5_C=0.9_K3=0.9_pattern_5  0.5  0.5  0.9  0.9  pattern_5   

                                              jacobian         